Trainning

Music2Vec

In [1]:
import os
import torch
import torchaudio
from torch.utils.data import Dataset
from transformers import AutoFeatureExtractor, Data2VecAudioForSequenceClassification, Trainer, TrainingArguments
import evaluate

# --------------------
# Dataset definition
# --------------------
class Music2VecDataset(Dataset):
    def __init__(self, root_dir, processor, target_sr=16000):
        self.files = []
        self.labels = []
        self.processor = processor  # This is actually the feature extractor
        self.target_sr = target_sr
        
        for fname in os.listdir(root_dir):
            if fname.lower().endswith(".wav"):
                path = os.path.join(root_dir, fname)
                label = 0 if fname.lower().startswith("real_") else 1
                self.files.append(path)
                self.labels.append(label)

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        path = self.files[idx]
        label = self.labels[idx]
        waveform, sr = torchaudio.load(path)
        # Mono
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)
        # Resample
        if sr != self.target_sr:
            waveform = torchaudio.functional.resample(waveform, sr, self.target_sr)
        waveform = waveform.squeeze().numpy()

        # Use self.processor instead of self.feature_extractor
        inputs = self.processor(waveform, sampling_rate=self.target_sr, return_tensors="pt", padding=True)
        inputs = {k: v.squeeze(0) for k, v in inputs.items()}
        inputs["labels"] = torch.tensor(label, dtype=torch.long)
        return inputs

# --------------------
# Load processor & model
# --------------------
processor = AutoFeatureExtractor.from_pretrained("m-a-p/music2vec-v1")

model = Data2VecAudioForSequenceClassification.from_pretrained(
    "m-a-p/music2vec-v1",
    num_labels=2
)

# --------------------
# Create datasets
# --------------------
train_dataset = Music2VecDataset("dataset/Train", processor)
valid_dataset = Music2VecDataset("dataset/Valid", processor)
test_dataset  = Music2VecDataset("dataset/Test", processor)

# --------------------
# Metrics
# --------------------
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return {
        "accuracy": accuracy_metric.compute(predictions=preds, references=labels)["accuracy"],
        "f1": f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"]
    }

# --------------------
# Training
# --------------------
training_args = TrainingArguments(
    output_dir="./music2vec-finetuned",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=100,
    fp16=torch.cuda.is_available(),
    push_to_hub=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    tokenizer=processor,
    compute_metrics=compute_metrics
)

trainer.train()

# --------------------
# Evaluation on test set
# --------------------
metrics = trainer.evaluate(test_dataset)
print(metrics)

c:\Users\j3n50\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:795: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of Data2VecAudioForSequenceClassification were not initialized from the model checkpoint at m-a-p/music2vec-v1 and are newly initialized: ['classifier.bias', 'classifier.weight', 'projector.weight', 'projector.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
c:\Users\j3n50\AppData\Local\Programs\Python\Python310\lib\site-packages\transformers\trainer.py:621: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler()
c:\Users\j3n50\AppData\Local\Programs\Python\Python310\lib\site-packag

  0%|          | 0/8850 [00:00<?, ?it/s]

c:\Users\j3n50\AppData\Local\Programs\Python\Python310\lib\site-packages\transformers\trainer.py:2610: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  else torch.cuda.amp.autocast(cache_enabled=cache_enabled, dtype=self.amp_dtype)


{'loss': 0.432, 'learning_rate': 2.968135593220339e-05, 'epoch': 0.03}
{'loss': 0.3613, 'learning_rate': 2.9345762711864408e-05, 'epoch': 0.07}
{'loss': 0.261, 'learning_rate': 2.900677966101695e-05, 'epoch': 0.1}
{'loss': 0.2138, 'learning_rate': 2.8671186440677967e-05, 'epoch': 0.14}
{'loss': 0.1451, 'learning_rate': 2.833220338983051e-05, 'epoch': 0.17}
{'loss': 0.2721, 'learning_rate': 2.7993220338983053e-05, 'epoch': 0.2}
{'loss': 0.2647, 'learning_rate': 2.7654237288135592e-05, 'epoch': 0.24}
{'loss': 0.2202, 'learning_rate': 2.7315254237288135e-05, 'epoch': 0.27}
{'loss': 0.161, 'learning_rate': 2.6979661016949155e-05, 'epoch': 0.31}
{'loss': 0.2196, 'learning_rate': 2.6640677966101698e-05, 'epoch': 0.34}
{'loss': 0.2027, 'learning_rate': 2.6301694915254237e-05, 'epoch': 0.37}
{'loss': 0.2335, 'learning_rate': 2.596271186440678e-05, 'epoch': 0.41}
{'loss': 0.1766, 'learning_rate': 2.5627118644067797e-05, 'epoch': 0.44}
{'loss': 0.146, 'learning_rate': 2.528813559322034e-05, 'epo

  0%|          | 0/338 [00:00<?, ?it/s]

...

  0%|          | 0/338 [00:00<?, ?it/s]

{'eval_loss': 0.5746986865997314, 'eval_accuracy': 0.9347664936990363, 'eval_f1': 0.9110099610792268, 'eval_runtime': 45.1497, 'eval_samples_per_second': 29.878, 'eval_steps_per_second': 7.486, 'epoch': 3.0}
{'train_runtime': 3020.7578, 'train_samples_per_second': 11.719, 'train_steps_per_second': 2.93, 'train_loss': 0.09495472580699597, 'epoch': 3.0}


c:\Users\j3n50\AppData\Local\Programs\Python\Python310\lib\site-packages\transformers\trainer.py:2610: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  else torch.cuda.amp.autocast(cache_enabled=cache_enabled, dtype=self.amp_dtype)


  0%|          | 0/341 [00:00<?, ?it/s]

{'eval_loss': 0.2542083263397217, 'eval_accuracy': 0.9684519442406456, 'eval_f1': 0.9623689777940435, 'eval_runtime': 47.6327, 'eval_samples_per_second': 28.615, 'eval_steps_per_second': 7.159, 'epoch': 3.0}


In [ ]:
# ==== TEST-ONLY, STREAMED EVALUATION FOR LONG AUDIO ====
# - Loads best/last checkpoint from ./music2vec-finetuned
# - Slides 10s windows with 50% overlap over each file
# - Averages logits per file -> prediction
# - Outputs classification report, confusion matrix, AUC
import os, json, glob, re, math
import numpy as np
import pandas as pd
import torch, torchaudio
from transformers import AutoFeatureExtractor, Data2VecAudioForSequenceClassification
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# --------------------
# Config
# --------------------
CKPT_BASE   = "./music2vec-finetuned"   # where your previous training saved
TEST_DIR    = "dataset/Test"            # folder with Real_*.wav / Fake_*.wav
TARGET_SR   = 16000
CHUNK_SEC   = 10.0   # window length
HOP_FRAC    = 0.5    # 50% overlap
BATCH_WIN   = 4      # how many windows to batch per forward pass

torch.set_num_threads(1)  # be gentle on Windows notebooks

# --------------------
# Utils
# --------------------
def resolve_checkpoint_dir(base_dir="./music2vec-finetuned"):
    if not os.path.isdir(base_dir):
        raise RuntimeError(f"Checkpoint dir not found: {base_dir}")
    state_path = os.path.join(base_dir, "trainer_state.json")
    if os.path.isfile(state_path):
        try:
            with open(state_path, "r", encoding="utf-8") as f:
                state = json.load(f)
            best = state.get("best_model_checkpoint", None)
            if best and os.path.isdir(best):
                return best
        except Exception:
            pass
    ckpts = [p for p in glob.glob(os.path.join(base_dir, "checkpoint-*")) if os.path.isdir(p)]
    if ckpts:
        def step_num(p):
            m = re.search(r"checkpoint-(\d+)", p)
            return int(m.group(1)) if m else -1
        ckpts.sort(key=step_num)
        return ckpts[-1]
    return base_dir

def softmax_np(z):
    z = z - z.max(axis=1, keepdims=True)
    ez = np.exp(z)
    return ez / ez.sum(axis=1, keepdims=True)

def list_test_files(test_dir):
    files, labels = [], []
    for fname in sorted(os.listdir(test_dir)):
        if fname.lower().endswith(".wav"):
            label = 0 if fname.lower().startswith("real_") else 1
            files.append(os.path.join(test_dir, fname))
            labels.append(label)
    if not files:
        raise RuntimeError(f"No .wav files in {test_dir} (expect names Real_*.wav / Fake_*.wav)")
    return files, labels

# --------------------
# Load model & processor
# --------------------
ckpt_dir = resolve_checkpoint_dir(CKPT_BASE)
print(f"Loading checkpoint from: {ckpt_dir}")

try:
    processor = AutoFeatureExtractor.from_pretrained(ckpt_dir)
except Exception:
    processor = AutoFeatureExtractor.from_pretrained("m-a-p/music2vec-v1")

model = Data2VecAudioForSequenceClassification.from_pretrained(ckpt_dir)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

# --------------------
# Collect test files
# --------------------
X, y_true = list_test_files(TEST_DIR)
print(f"Found {len(X)} test files.")

# --------------------
# Evaluate per file with sliding windows
# --------------------
chunk_len = int(CHUNK_SEC * TARGET_SR)
hop_len   = int(HOP_FRAC * CHUNK_SEC * TARGET_SR)

all_logits = []
for idx, path in enumerate(X, 1):
    # Load & resample mono
    wav, sr = torchaudio.load(path)
    if wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True)
    if sr != TARGET_SR:
        wav = torchaudio.functional.resample(wav, sr, TARGET_SR)
    x = wav.squeeze(0)  # [T]

    # Build windows (10s, 50% hop). If very short, use single padded window.
    if x.numel() <= chunk_len:
        windows = [torch.nn.functional.pad(x, (0, chunk_len - x.numel()))]
    else:
        starts = list(range(0, x.numel() - chunk_len + 1, hop_len if hop_len > 0 else chunk_len))
        windows = [x[s:s+chunk_len] for s in starts]
        # Ensure last chunk covers the tail
        if starts and (starts[-1] + chunk_len < x.numel()):
            tail = x[-chunk_len:]
            windows.append(tail)

    # Batched inference over windows
    file_logits = []
    with torch.no_grad():
        for b in range(0, len(windows), BATCH_WIN):
            batch_wins = windows[b:b+BATCH_WIN]
            feats = processor(
                [w.numpy() for w in batch_wins],
                sampling_rate=TARGET_SR,
                return_tensors="pt",
                padding=True
            )
            feats = {k: v.to(device) for k, v in feats.items() if k != "labels"}
            out = model(**feats).logits  # [B, 2]
            file_logits.append(out.cpu())
    if file_logits:
        file_logits = torch.cat(file_logits, dim=0)   # [Nwin, 2]
        mean_logits = file_logits.mean(dim=0, keepdim=True).numpy()  # [1, 2]
    else:
        # Shouldn't happen, but fallback
        mean_logits = np.zeros((1, 2), dtype=np.float32)

    all_logits.append(mean_logits)

    # Progress line
    dur_sec = x.numel() / TARGET_SR
    nwin = len(windows)
    print(f"[{idx}/{len(X)}] {os.path.basename(path)}  dur={dur_sec/60:.1f} min  windows={nwin}")

# Stack per-file logits
logits = np.concatenate(all_logits, axis=0)  # [Nfile, 2]
y_true = np.array(y_true)
y_pred = logits.argmax(axis=-1)
probs  = softmax_np(logits)
p_fake = probs[:, 1]  # prob of class 1 (Fake)

# --------------------
# Reports
# --------------------
target_names = ["Real (0)", "Fake (1)"]
report_str = classification_report(y_true, y_pred, target_names=target_names, digits=4)
print("\n=== Classification Report (Test) ===")
print(report_str)

cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
cm_df = pd.DataFrame(cm, index=["True Real (0)", "True Fake (1)"],
                        columns=["Pred Real (0)", "Pred Fake (1)"])
print("\n=== Confusion Matrix (Test) ===")
print(cm_df)

try:
    auc = roc_auc_score(y_true, p_fake)
    print(f"\nAUC (Fake=positive): {auc:.4f}")
except ValueError:
    print("\nAUC could not be computed (need both classes present).")

# --------------------
# Save artifacts
# --------------------
with open("classification_report_test.txt", "w", encoding="utf-8") as f:
    f.write(report_str)

pd.DataFrame(classification_report(y_true, y_pred, target_names=target_names, output_dict=True)).transpose()\
  .to_csv("classification_report_test.csv", index=True)
cm_df.to_csv("confusion_matrix_test.csv", index=True)

print("\nSaved files:")
print(" - classification_report_test.txt")
print(" - classification_report_test.csv")
print(" - confusion_matrix_test.csv")


In [3]:
# ==== VISUALIZE PER-FILE RESULTS ====

# Build a DataFrame for visualization
results_df = pd.DataFrame({
    "Filename": [os.path.basename(p) for p in X],
    "TrueLabel": ["Real" if t == 0 else "Fake" for t in y_true],
    "PredLabel": ["Real" if p == 0 else "Fake" for p in y_pred],
    "Prob_Fake": p_fake
})

# Sort by probability of Fake (descending) to see strongest predictions first
results_df = results_df.sort_values(by="Prob_Fake", ascending=False).reset_index(drop=True)

# Print the full table
print("\n=== Per-file Predictions ===")
print(results_df.to_string(index=False))

# Save to CSV for later
results_df.to_csv("per_file_predictions.csv", index=False)

print("\nSaved per-file predictions to per_file_predictions.csv")



=== Per-file Predictions ===
                                                                                                         Filename TrueLabel PredLabel  Prob_Fake
                                                          Fake_TOP MUSIC PLAYLIST GOOD VIBES [kvx5ckMi8Oo]_74.wav      Fake      Fake   0.999963
                                                          Fake_TOP MUSIC PLAYLIST GOOD VIBES [kvx5ckMi8Oo]_68.wav      Fake      Fake   0.999962
                                                          Fake_TOP MUSIC PLAYLIST GOOD VIBES [kvx5ckMi8Oo]_70.wav      Fake      Fake   0.999962
                                                          Fake_TOP MUSIC PLAYLIST GOOD VIBES [kvx5ckMi8Oo]_36.wav      Fake      Fake   0.999962
                                           Fake_TOP MUSIC PLAYLIST ACCOUSTIC AND CHILL VIBES [pC8OCPuW_uA]_57.wav      Fake      Fake   0.999962
                                                          Fake_TOP MUSIC PLAYLIST GOOD VIBES [kvx5ck

In [4]:
import gc, torch

# Safely delete model & optimizer if they exist
for var in ["model", "optimizer"]:
    if var in globals():
        del globals()[var]

# Force garbage collection & free GPU cache
gc.collect()
torch.cuda.empty_cache()


In [5]:
import tensorflow as tf
from tensorflow.keras import backend as K

K.clear_session()
tf.config.experimental.reset_memory_stats('GPU:0')


PASST

In [1]:
# !pip install transformers torchaudio datasets accelerate evaluate

import os
import torch
import torchaudio
from torch.utils.data import Dataset
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification, Trainer, TrainingArguments
import evaluate

# --------------------
# Paths
# --------------------
TRAIN_DIR = "D:\Thesis\Song\dataset\Train"
VAL_DIR   = "D:\Thesis\Song\dataset\Valid"
TEST_DIR  = "D:\Thesis\Song\dataset\Test"

# --------------------
# Load feature extractor & model
# --------------------
MODEL_ID = "MIT/ast-finetuned-audioset-10-10-0.4593"
feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_ID)
model = AutoModelForAudioClassification.from_pretrained(
    MODEL_ID,
    num_labels=2,
    ignore_mismatched_sizes=True,  # Ignore final layer size mismatch

)

TARGET_SR = feature_extractor.sampling_rate  # PaSST expects 32000 Hz

# --------------------
# Dataset
# --------------------
class PaSSTDataset(Dataset):
    def __init__(self, root_dir, feature_extractor, target_sr):
        self.files = []
        self.labels = []
        self.feature_extractor = feature_extractor
        self.target_sr = target_sr

        min_len = int(0.025 * target_sr)  # 25ms minimum window size

        for fname in sorted(os.listdir(root_dir)):
            if fname.lower().endswith(".wav"):
                path = os.path.join(root_dir, fname)
                label = 0 if fname.lower().startswith("real_") else 1

                # Load just to check length
                waveform, sr = torchaudio.load(path)
                if waveform.shape[0] > 1:
                    waveform = waveform.mean(dim=0, keepdim=True)
                if sr != target_sr:
                    waveform = torchaudio.functional.resample(waveform, sr, target_sr)
                waveform = waveform.squeeze()

                # Keep only if long enough
                if waveform.shape[-1] >= min_len:
                    self.files.append(path)
                    self.labels.append(label)

        print(f"Loaded {len(self.files)} files from {root_dir} (dropped too-short ones)")

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        path = self.files[idx]
        label = self.labels[idx]
        waveform, sr = torchaudio.load(path)

        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)
        if sr != self.target_sr:
            waveform = torchaudio.functional.resample(waveform, sr, self.target_sr)
        waveform = waveform.squeeze().numpy()

        inputs = self.feature_extractor(
            waveform,
            sampling_rate=self.target_sr,
            return_tensors="pt",
            padding=True
        )
        inputs = {k: v.squeeze(0) for k, v in inputs.items()}
        inputs["labels"] = torch.tensor(label, dtype=torch.long)
        return inputs


# --------------------
# Build datasets
# --------------------
train_dataset = PaSSTDataset(TRAIN_DIR, feature_extractor, TARGET_SR)
val_dataset   = PaSSTDataset(VAL_DIR,   feature_extractor, TARGET_SR)
test_dataset  = PaSSTDataset(TEST_DIR,  feature_extractor, TARGET_SR)

# --------------------
# Metrics
# --------------------
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")
roc_metric = evaluate.load("roc_auc")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    probs = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
    return {
        "accuracy": accuracy_metric.compute(predictions=preds, references=labels)["accuracy"],
        "f1_macro": f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"],
        "auroc": roc_metric.compute(prediction_scores=probs, references=labels)["roc_auc"]
    }

# --------------------
# Training arguments
# --------------------
training_args = TrainingArguments(
    output_dir="./passt-light-finetuned",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="auroc",
    greater_is_better=True,
    learning_rate=3e-5,
    per_device_train_batch_size=6,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=100,
    fp16=torch.cuda.is_available(),
    push_to_hub=False,
    seed=42
)

# --------------------
# Trainer
# --------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=feature_extractor,
    compute_metrics=compute_metrics
)

# --------------------
# Train
# --------------------
trainer.train()

# --------------------
# Evaluate on test set
# --------------------
test_metrics = trainer.evaluate(test_dataset)
print("\n=== Test set metrics ===")
for k, v in test_metrics.items():
    if k.startswith("eval_"):
        print(f"{k[5:]}: {v:.4f}")


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Some weights of ASTForAudioClassification were not initialized from the model checkpoint at MIT/ast-finetuned-audioset-10-10-0.4593 and are newly initialized because the shapes did not match:
- classifier.dense.bias: found shape torch.Size([527]) in the checkpoint and torch.Size([2]) in the model instantiated
- classifier.dense.weight: found shape torch.Size([527, 768]) in the checkpoint and torch.Size([2, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loaded 14315 files from D:\Thesis\Song\dataset\Train (dropped too-short ones)
Loaded 1457 files from D:\Thesis\Song\dataset\Valid (dropped too-short ones)
Loaded 1718 files from D:\Thesis\Song\dataset\Test (dropped too-short ones)


C:\Users\j3n50\AppData\Local\Temp\ipykernel_16060\1340318888.py:137: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,Auroc
1,0.021600,0.030764,0.994509,0.994479,0.999812
2,0.015000,0.019830,0.995882,0.995860,0.999936
3,0.000000,0.029939,0.994509,0.994481,0.999924



=== Test set metrics ===
loss: 0.3430
accuracy: 0.9336
f1_macro: 0.9325
auroc: 0.9981
runtime: 47.7145
samples_per_second: 36.0060
steps_per_second: 9.0120


In [2]:
# ==== TEST-ONLY EVALUATION (PaSST) FOR ALREADY-CHUNKED TEST FILES ====
import os, json, glob, re
import numpy as np
import pandas as pd
import torch, torchaudio
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# --------------------
# Config
# --------------------
CKPT_BASE = "D:\Thesis\Song\TrainClassification\passt-light-finetuned\checkpoint-7158"   # folder where you trained PaSST
TEST_DIR  = "D:\Thesis\Song\dataset\Test"              # contains Real_*.wav / Fake_*.wav (already chunked)
BATCH_SIZE = 48                         # increase/decrease based on VRAM/CPU
torch.set_num_threads(1)

# --------------------
# Helpers
# --------------------
def resolve_checkpoint_dir(base_dir="./passt-light-finetuned"):
    """Use best_model_checkpoint if present; else last checkpoint-*; else base_dir."""
    if not os.path.isdir(base_dir):
        raise RuntimeError(f"Checkpoint dir not found: {base_dir}")
    state_path = os.path.join(base_dir, "trainer_state.json")
    if os.path.isfile(state_path):
        try:
            with open(state_path, "r", encoding="utf-8") as f:
                state = json.load(f)
            best = state.get("best_model_checkpoint", None)
            if best and os.path.isdir(best):
                return best
        except Exception:
            pass
    ckpts = [p for p in glob.glob(os.path.join(base_dir, "checkpoint-*")) if os.path.isdir(p)]
    if ckpts:
        def step_num(p):
            m = re.search(r"checkpoint-(\d+)", p)
            return int(m.group(1)) if m else -1
        ckpts.sort(key=step_num)
        return ckpts[-1]
    return base_dir

def softmax_np(z):
    z = z - z.max(axis=1, keepdims=True)
    ez = np.exp(z)
    return ez / ez.sum(axis=1, keepdims=True)

def list_labeled_files(test_dir):
    files, labels = [], []
    for fname in sorted(os.listdir(test_dir)):
        if fname.lower().endswith(".wav"):
            label = 0 if fname.lower().startswith("real_") else 1
            files.append(os.path.join(test_dir, fname))
            labels.append(label)
    if not files:
        raise RuntimeError(f"No .wav files in {test_dir} (expected Real_*.wav / Fake_*.wav)")
    return files, np.array(labels, dtype=int)

# --------------------
# Load model + feature extractor
# --------------------
ckpt_dir = resolve_checkpoint_dir(CKPT_BASE)
print(f"Loading checkpoint from: {ckpt_dir}")

# Load feature extractor (prefer from checkpoint directory)
try:
    feature_extractor = AutoFeatureExtractor.from_pretrained(ckpt_dir)
except Exception:
    feature_extractor = AutoFeatureExtractor.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593")

TARGET_SR = getattr(feature_extractor, "sampling_rate", 32000)  # AST uses 32k
model = AutoModelForAudioClassification.from_pretrained(ckpt_dir)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

# --------------------
# Collect test files
# --------------------
X, y_true = list_labeled_files(TEST_DIR)
print(f"Found {len(X)} test chunks.")

# --------------------
# Batched inference (no sliding windows)
# --------------------
logits_list = []
for i in range(0, len(X), BATCH_SIZE):
    batch_paths = X[i:i+BATCH_SIZE]
    waves = []
    for p in batch_paths:
        wav, sr = torchaudio.load(p)
        if wav.shape[0] > 1:
            wav = wav.mean(dim=0, keepdim=True)
        if sr != TARGET_SR:
            wav = torchaudio.functional.resample(wav, sr, TARGET_SR)
        waves.append(wav.squeeze(0).numpy())

    with torch.no_grad():
        feats = feature_extractor(
            waves,
            sampling_rate=TARGET_SR,
            return_tensors="pt",
            padding=True
        )
        feats = {k: v.to(device) for k, v in feats.items()}
        out = model(**feats).logits  # [B, 2]
        logits_list.append(out.cpu().numpy())

    print(f"Processed {min(i+BATCH_SIZE, len(X))}/{len(X)}")

logits = np.concatenate(logits_list, axis=0)  # [N, 2]
y_pred = logits.argmax(axis=-1)
probs  = softmax_np(logits)
p_fake = probs[:, 1]  # probability of class 1 (Fake)

# --------------------
# Reports + CSVs
# --------------------
target_names = ["Real (0)", "Fake (1)"]

rep_str  = classification_report(y_true, y_pred, target_names=target_names, digits=4)
rep_dict = classification_report(y_true, y_pred, target_names=target_names, output_dict=True)
rep_df   = pd.DataFrame(rep_dict).transpose()
print("\n=== Classification Report (Test) ===")
print(rep_str)

cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
cm_df = pd.DataFrame(cm, index=["True Real (0)", "True Fake (1)"],
                        columns=["Pred Real (0)", "Pred Fake (1)"])
print("\n=== Confusion Matrix (Test) ===")
print(cm_df)

try:
    auc = roc_auc_score(y_true, p_fake)
    print(f"\nAUC (Fake=positive): {auc:.4f}")
except ValueError:
    print("\nAUC could not be computed (need both classes present).")

per_file_df = pd.DataFrame({
    "Filename": [os.path.basename(p) for p in X],
    "TrueLabel": ["Real" if t == 0 else "Fake" for t in y_true],
    "PredLabel": ["Real" if p == 0 else "Fake" for p in y_pred],
    "Prob_Fake": p_fake
}).sort_values("Prob_Fake", ascending=False).reset_index(drop=True)

# Save to the training folder
os.makedirs(CKPT_BASE, exist_ok=True)
rep_df.to_csv(os.path.join(CKPT_BASE, "classification_report_test.csv"), index=True)
cm_df.to_csv(os.path.join(CKPT_BASE, "confusion_matrix_test.csv"), index=True)
per_file_df.to_csv(os.path.join(CKPT_BASE, "per_file_predictions.csv"), index=False)

with open(os.path.join(CKPT_BASE, "classification_report_test.txt"), "w", encoding="utf-8") as f:
    f.write(rep_str)

print("\nSaved:")
print(" -", os.path.join(CKPT_BASE, "classification_report_test.csv"))
print(" -", os.path.join(CKPT_BASE, "confusion_matrix_test.csv"))
print(" -", os.path.join(CKPT_BASE, "per_file_predictions.csv"))
print(" -", os.path.join(CKPT_BASE, "classification_report_test.txt"))


Loading checkpoint from: ./passt-light-finetuned\checkpoint-4772
Found 1718 test chunks.
Processed 48/1718
Processed 96/1718
Processed 144/1718
Processed 192/1718
Processed 240/1718
Processed 288/1718
Processed 336/1718
Processed 384/1718
Processed 432/1718
Processed 480/1718
Processed 528/1718
Processed 576/1718
Processed 624/1718
Processed 672/1718
Processed 720/1718
Processed 768/1718
Processed 816/1718
Processed 864/1718
Processed 912/1718
Processed 960/1718
Processed 1008/1718
Processed 1056/1718
Processed 1104/1718
Processed 1152/1718
Processed 1200/1718
Processed 1248/1718
Processed 1296/1718
Processed 1344/1718
Processed 1392/1718
Processed 1440/1718
Processed 1488/1718
Processed 1536/1718
Processed 1584/1718
Processed 1632/1718
Processed 1680/1718
Processed 1718/1718

=== Classification Report (Test) ===
              precision    recall  f1-score   support

    Real (0)     0.8893    1.0000    0.9414       916
    Fake (1)     1.0000    0.8579    0.9235       802

    accurac